In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive, files

In [ ]:
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.copy("/content/kaggle/kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle token")

Kaggle token


In [ ]:
import subprocess
result = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "batuhankalem/turkishlaw-dataset-for-llm-finetuning",
     "-p", "kaggle_raw/", "--unzip"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)

STDOUT: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata

STDERR: 
Return code: 1


In [ ]:
import kagglehub

path = kagglehub.dataset_download("batuhankalem/turkish-law-dataset-for-llm-finetuning")
print("Path:", path)
print("Dosyalar:", os.listdir(path))

100%|██████████| 3.55M/3.55M [00:00<00:00, 146MB/s]

Extracting files...


Path: /root/.cache/kagglehub/datasets/batuhankalem/turkish-law-dataset-for-llm-finetuning/versions/1
Dosyalar: ['turkish_law_dataset.csv']


In [ ]:
RAW_FILE = os.path.join(path, "turkish_law_dataset.csv")

try:
    df = pd.read_csv(RAW_FILE, encoding="utf-8")
    print("file okundu (utf-8)")
except Exception:
    df = pd.read_csv(RAW_FILE, encoding="latin-1")
    print("file okundu (latin-1)")

file okundu (utf-8)


In [ ]:
#veri incelemesi
print("\n" + "="*60)
print("HAM VERİ RAPORU")
print("="*60)
print(f"Toplam satır    : {len(df)}")
print(f"Sütunlar        : {list(df.columns)}")
print(f"\nEksik değerler:\n{df.isnull().sum()}")
print(f"\nUnique değerler:\n{df.nunique()}")
print(f"\nİlk 2 satır:")
print(df.head(2).to_string())


HAM VERİ RAPORU
Toplam satır    : 13954
Sütunlar        : ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'Score']

Eksik değerler:
soru         0
cevap        0
veri türü    0
kaynak       0
context      0
Score        0
dtype: int64

Unique değerler:
soru         12885
cevap        13373
veri türü        1
kaynak           8
context        240
Score           12
dtype: int64

İlk 2 satır:
                                                                                        soru                                                                                                                                                                                                                                                                         cevap veri türü                         kaynak                                                                                                                                                                                                      

In [ ]:
# Context boşluk kontrolü
#veride atılacakları karar vermek için
print("Context boş olanlar:", (df["context"].str.strip() == "").sum())

# Kaynak dağılımı
print("\nKaynak dağılımı:")
print(df["kaynak"].value_counts())
print("\nOranlar:")
print(df["kaynak"].value_counts(normalize=True).round(3) * 100)

print("Kısa soru (< 15 kar):", (df["soru"].str.len() < 15).sum())
print("Kısa cevap (< 20 kar):", (df["cevap"].str.len() < 20).sum())

Context boş olanlar: 0

Kaynak dağılımı:
kaynak
Türk Ceza Kanunu                 3738
Türk Medeni Kanunu               3399
Ceza Muhakemesi Kanunu           2074
Türk Borçlar Kanunu              1791
Türkiye Cumhuriyeti Anayasası    1488
Türkiye Cumhuriyeti İş Kanunu     822
Türk Bayrağı Tüzüğü               388
Bilgi Edinme Kanunu               254
Name: count, dtype: int64

Oranlar:
kaynak
Türk Ceza Kanunu                 26.8
Türk Medeni Kanunu               24.4
Ceza Muhakemesi Kanunu           14.9
Türk Borçlar Kanunu              12.8
Türkiye Cumhuriyeti Anayasası    10.7
Türkiye Cumhuriyeti İş Kanunu     5.9
Türk Bayrağı Tüzüğü               2.8
Bilgi Edinme Kanunu               1.8
Name: proportion, dtype: float64
Kısa soru (< 15 kar): 3
Kısa cevap (< 20 kar): 2


In [ ]:
#normalize et(boşlukları kaldır, büyük harften küçüğe vb.)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
print(f"\nNormalize edilmiş sütunlar: {list(df.columns)}")


Normalize edilmiş sütunlar: ['soru', 'cevap', 'veri_türü', 'kaynak', 'context', 'score']


In [ ]:
#gereksiz stünları düşürme
#"veri türü" sütunu 1 uniqe valuesi olduğu için gereksiz
drop_cols = []
for col in df.columns:
    if col == "score":
        continue  # Score'u filtrede kullanacağız, şimdi atma
    if df[col].nunique() == 1:
        print(f"  Düşürülüyor (1 unique value): '{col}' → değer: {df[col].iloc[0]}")
        drop_cols.append(col)

df = df.drop(columns=drop_cols)
print(f"\nKalan sütunlar: {list(df.columns)}")

  Düşürülüyor (1 unique value): 'veri_türü' → değer: hukuk

Kalan sütunlar: ['soru', 'cevap', 'kaynak', 'context', 'score']


In [ ]:
#sütun adlarını standartlaştırma

soru_col    = next((c for c in df.columns if "soru" in c or "question" in c), None)
cevap_col   = next((c for c in df.columns if "cevap" in c or "answer" in c), None)
kaynak_col  = next((c for c in df.columns if "kaynak" in c or "source" in c), None)
context_col = next((c for c in df.columns if "context" in c), None)

print(f"\nEşleşen sütunlar:")
print(f"  soru    → {soru_col}")
print(f"  cevap   → {cevap_col}")
print(f"  kaynak  → {kaynak_col}")
print(f"  context → {context_col}")

rename_map = {}
if soru_col:    rename_map[soru_col]    = "soru"
if cevap_col:   rename_map[cevap_col]   = "cevap"
if kaynak_col:  rename_map[kaynak_col]  = "kaynak"
if context_col: rename_map[context_col] = "context"

df = df.rename(columns=rename_map)


Eşleşen sütunlar:
  soru    → soru
  cevap   → cevap
  kaynak  → kaynak
  context → context


In [ ]:
#null satırları silme
onceki = len(df)
df = df.dropna(subset=["soru", "cevap"])
df["soru"]  = df["soru"].astype(str).str.strip()
df["cevap"] = df["cevap"].astype(str).str.strip()
df = df[(df["soru"] != "") & (df["cevap"] != "")]
print(f"\nBoş satır silme : {onceki} → {len(df)} ({onceki - len(df)} silindi)")


Boş satır silme : 13954 → 13954 (0 silindi)


In [ ]:
#düşük skorları silme
MIN_SCORE = 5  # 0-4 arası şüpheli, <0 açıkça kötü

onceki = len(df)
print(f"\nScore filtresi öncesi dağılım:\n{df['score'].value_counts().sort_index()}")

df = df[df["score"] >= MIN_SCORE]
print(f"\nScore filtresi (≥{MIN_SCORE}): {onceki} → {len(df)} ({onceki - len(df)} silindi)")

# Score sütununu artık düşür (fine-tuning için gerekmiyor)
df = df.drop(columns=["score"])


Score filtresi öncesi dağılım:
score
-1       17
 0        4
 1       16
 2       44
 3       39
 4       22
 5       47
 6      168
 7     1094
 8     7866
 9     3784
 10     853
Name: count, dtype: int64

Score filtresi (≥5): 13954 → 13812 (142 silindi)


In [ ]:
#kısa satırları filtreleme
MIN_SORU  = 15
MIN_CEVAP = 20

onceki = len(df)
df = df[df["soru"].str.len() >= MIN_SORU]
df = df[df["cevap"].str.len() >= MIN_CEVAP]
print(f"Çok kısa filtre : {onceki} → {len(df)} ({onceki - len(df)} silindi)")

Çok kısa filtre : 13812 → 13808 (4 silindi)


In [ ]:
#duaplicate temizleme
onceki = len(df)
df = df.drop_duplicates(subset=["soru", "cevap"])
print(f"Tam duplicate   : {onceki} → {len(df)} ({onceki - len(df)} silindi)")

onceki = len(df)
df["_len"] = df["cevap"].str.len()
df = df.sort_values("_len", ascending=False)
df = df.drop_duplicates(subset=["soru"], keep="first")
df = df.drop(columns=["_len"])
print(f"Soru duplicate  : {onceki} → {len(df)} ({onceki - len(df)} silindi)")

df = df.reset_index(drop=True)


Tam duplicate   : 13808 → 13505 (303 silindi)
Soru duplicate  : 13505 → 12765 (740 silindi)


In [ ]:
#context sütunu temizleme gereksiz boşluklar vb
if "context" in df.columns:
    df["context"] = df["context"].fillna("").astype(str)
    df["context"] = df["context"].str.replace(r"BAŞLANGIÇ\s*\[\d+\]", "", regex=True)
    df["context"] = df["context"].str.replace(r"\s+", " ", regex=True).str.strip()
    print(f"\nContext temizlendi. Örnek: {df['context'].iloc[0][:100]}")


Context temizlendi. Örnek: BİRİNCİ KİTAP Genel Hükümler DÖRDÜNCÜ KISIM Koruma Tedbirleri İKİNCİ BÖLÜM Tutuklama Tutuklama neden


In [ ]:
# Ham context'te hâlâ sorunlu karakter var mı?
sample = df["context"].iloc[0]
print(repr(sample[:300]))

'BİRİNCİ KİTAP Genel Hükümler DÖRDÜNCÜ KISIM Koruma Tedbirleri İKİNCİ BÖLÜM Tutuklama Tutuklama nedenleri Madde 100 – (1) Kuvvetli suç şüphesinin varlığını gösteren somut delillerin ve bir tutuklama nedeninin bulunması halinde, şüpheli veya sanık hakkında tutuklama kararı verilebilir. İşin önemi, ver'


In [ ]:
#temizleme raporu
print("\n" + "="*60)
print("TEMİZ VERİ RAPORU")
print("="*60)
print(f"Toplam örnek    : {len(df)}")
if "kaynak" in df.columns:
    print(f"\nKaynak dağılımı:\n{df['kaynak'].value_counts()}")
print(f"\nSoru uzunluğu  : min={df['soru'].str.len().min()}, "
      f"ort={df['soru'].str.len().mean():.0f}, "
      f"max={df['soru'].str.len().max()}")
print(f"Cevap uzunluğu : min={df['cevap'].str.len().min()}, "
      f"ort={df['cevap'].str.len().mean():.0f}, "
      f"max={df['cevap'].str.len().max()}")


TEMİZ VERİ RAPORU
Toplam örnek    : 12765

Kaynak dağılımı:
kaynak
Türk Ceza Kanunu                 3340
Türk Medeni Kanunu               3062
Ceza Muhakemesi Kanunu           1921
Türk Borçlar Kanunu              1644
Türkiye Cumhuriyeti Anayasası    1442
Türkiye Cumhuriyeti İş Kanunu     765
Türk Bayrağı Tüzüğü               345
Bilgi Edinme Kanunu               246
Name: count, dtype: int64

Soru uzunluğu  : min=16, ort=83, max=374
Cevap uzunluğu : min=28, ort=327, max=4573


In [ ]:
#train / test script
if "kaynak" in df.columns:
    kaynak_counts   = df["kaynak"].value_counts()
    valid_kaynaklar = kaynak_counts[kaynak_counts >= 10].index
    df_stratify     = df[df["kaynak"].isin(valid_kaynaklar)]
    df_rest         = df[~df["kaynak"].isin(valid_kaynaklar)]

    train_s, test_s = train_test_split(
        df_stratify, test_size=0.1, random_state=42,
        stratify=df_stratify["kaynak"]
    )
    train_df = pd.concat([train_s, df_rest], ignore_index=True)
    test_df  = test_s
else:
    train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

print(f"\nTrain : {len(train_df)} örnek")
print(f"Test  : {len(test_df)} örnek")


Train : 11488 örnek
Test  : 1277 örnek


In [ ]:
#kaydetme
os.makedirs("cleaned_data", exist_ok=True)

train_df.to_json("cleaned_data/kaggle_train_clean.jsonl",
                 orient="records", lines=True, force_ascii=False)
test_df.to_json("cleaned_data/kaggle_test_clean.jsonl",
                orient="records", lines=True, force_ascii=False)

print("\n Colab'a kaydedildi:")
print(f"   cleaned_data/kaggle_train_clean.jsonl  ({len(train_df)} örnek)")
print(f"   cleaned_data/kaggle_test_clean.jsonl   ({len(test_df)} örnek)")

# Google Drive'a kopyala
print("\nGoogle Drive bağlanıyor...")
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/CENG493/cleaned_data"
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copy("cleaned_data/kaggle_train_clean.jsonl", f"{DRIVE_DIR}/kaggle_train_clean.jsonl")
shutil.copy("cleaned_data/kaggle_test_clean.jsonl",  f"{DRIVE_DIR}/kaggle_test_clean.jsonl")

print(f"\n Google Drive'a kaydedildi: {DRIVE_DIR}/")
print(f"   kaggle_train_clean.jsonl  ({len(train_df)} örnek)")
print(f"   kaggle_test_clean.jsonl   ({len(test_df)} örnek)")

print(f"\n{'='*60}")
print("ÖZET")
print(f"{'='*60}")
print(f"Ham veri         : 13.954 satır")
print(f"Score filtresi   : ≥{MIN_SCORE} (Gemini 1.5 Pro kalite skoru)")
print(f"Temiz train      : {len(train_df)}")
print(f"Temiz test       : {len(test_df)}")



 Colab'a kaydedildi:
   cleaned_data/kaggle_train_clean.jsonl  (11488 örnek)
   cleaned_data/kaggle_test_clean.jsonl   (1277 örnek)

Google Drive bağlanıyor...
Mounted at /content/drive

 Google Drive'a kaydedildi: /content/drive/MyDrive/CENG493/cleaned_data/
   kaggle_train_clean.jsonl  (11488 örnek)
   kaggle_test_clean.jsonl   (1277 örnek)

ÖZET
Ham veri         : 13.954 satır
Score filtresi   : ≥5 (Gemini 1.5 Pro kalite skoru)
Temiz train      : 11488
Temiz test       : 1277


HUGGİNG FACE


In [1]:
import os
import json
import pandas as pd
from datasets import load_dataset

In [2]:
print("Dataset yükleme")
ds = load_dataset("Renicames/turkish-law-chatbot")

print(f"Split'ler: {list(ds.keys())}")
for split_name, split_data in ds.items():
    print(f"  {split_name}: {len(split_data)} örnek | sütunlar: {split_data.column_names}")

Dataset yükleme


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/13354 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Split'ler: ['train', 'test']
  train: 13354 örnek | sütunlar: ['Soru', 'Cevap']
  test: 1500 örnek | sütunlar: ['Soru', 'Cevap']


In [3]:
#ham veri inceleme
print("\n" + "="*60)
print("HAM VERİ RAPORU")
print("="*60)

train_df = ds["train"].to_pandas()
test_df  = ds["test"].to_pandas()

# Sütun isimlerini normalize et
train_df.columns = [c.strip() for c in train_df.columns]
test_df.columns  = [c.strip() for c in test_df.columns]

# Sütun isimlerini bul
soru_col  = next((c for c in train_df.columns
                  if c.lower() in ["soru", "question", "input"]), None)
cevap_col = next((c for c in train_df.columns
                  if c.lower() in ["cevap", "answer", "output", "response"]), None)

print(f"Soru sütunu  → {soru_col}")
print(f"Cevap sütunu → {cevap_col}")
print(f"\nTrain: {len(train_df)} satır")
print(f"Test : {len(test_df)} satır")
print(f"\nEksik (train):\n{train_df.isnull().sum()}")

print(f"\nİlk 3 örnek (train):")
for i in range(3):
    row = train_df.iloc[i]
    print(f"\n  [{i}] SORU : {row[soru_col]}")
    print(f"      CEVAP: {row[cevap_col]}")


HAM VERİ RAPORU
Soru sütunu  → Soru
Cevap sütunu → Cevap

Train: 13354 satır
Test : 1500 satır

Eksik (train):
Soru     0
Cevap    0
dtype: int64

İlk 3 örnek (train):

  [0] SORU : Anayasa madde 1'e göre, türkiye'nin devlet şekli nedir
      CEVAP: Anayasa madde 1'e göre, türkiye'nin devlet şekli cumhuriyettir. bu madde, türkiye'nin yönetim biçiminin halkın egemenliğine dayandığını ve bu yönetim biçiminin cumhuriyet olduğunu belirler. cumhuriyet, halkın kendi kendini yönetme biçimi olarak kabul edilir ve türkiye cumhuriyeti'nin temel yönetim ilkesi olarak anayasal güvence altına alınmıştır.

  [1] SORU : Anayasa madde 1'de belirtilen cumhuriyetin tanımı nedir
      CEVAP: Anayasa madde 1'de belirtilen cumhuriyet, halkın egemenliğinin temsilcileri aracılığıyla yönetildiği bir devlet şeklidir. cumhuriyet rejiminde, devlet başkanı ve diğer yöneticiler halk tarafından seçilir ve belirli bir süre için görev yaparlar. bu yönetim biçimi, monarşi gibi kalıtsal yönetim biçimlerine karşıdır ve

In [5]:
#standatlaştırma
def standardize(df, soru_col, cevap_col):
    df = df.rename(columns={soru_col: "soru", cevap_col: "cevap"})
    # Sadece soru ve cevap kalsın
    keep = ["soru", "cevap"]
    extra = [c for c in df.columns if c not in keep]
    if extra:
        print(f"  Ek sütunlar korunuyor: {extra}")
        keep += extra
    return df[keep].copy()

train_df = standardize(train_df, soru_col, cevap_col)
test_df  = standardize(test_df,  soru_col, cevap_col)

In [6]:
#boş satır silme
def clean_nulls(df, label):
    onceki = len(df)
    df = df.dropna(subset=["soru", "cevap"])
    df["soru"]  = df["soru"].astype(str).str.strip()
    df["cevap"] = df["cevap"].astype(str).str.strip()
    df = df[(df["soru"] != "") & (df["cevap"] != "")]
    print(f"  [{label}] Boş satır silme: {onceki} → {len(df)} ({onceki - len(df)} silindi)")
    return df

train_df = clean_nulls(train_df, "train")
test_df  = clean_nulls(test_df,  "test")

  [train] Boş satır silme: 13354 → 13354 (0 silindi)
  [test] Boş satır silme: 1500 → 1500 (0 silindi)


In [7]:
#çok kısa soruları sil
MIN_SORU  = 15
MIN_CEVAP = 20

def filter_short(df, label):
    onceki = len(df)
    df = df[df["soru"].str.len() >= MIN_SORU]
    df = df[df["cevap"].str.len() >= MIN_CEVAP]
    print(f"  [{label}] Çok kısa filtre: {onceki} → {len(df)} ({onceki - len(df)} silindi)")
    return df

train_df = filter_short(train_df, "train")
test_df  = filter_short(test_df,  "test")


  [train] Çok kısa filtre: 13354 → 13115 (239 silindi)
  [test] Çok kısa filtre: 1500 → 1469 (31 silindi)


In [8]:
#duaplicate temizleme
def remove_duplicates(df, label):
    onceki = len(df)
    df = df.drop_duplicates(subset=["soru", "cevap"])
    sonra1 = len(df)

    # Aynı soru → en uzun cevabı koru
    df["_len"] = df["cevap"].str.len()
    df = df.sort_values("_len", ascending=False)
    df = df.drop_duplicates(subset=["soru"], keep="first")
    df = df.drop(columns=["_len"])
    sonra2 = len(df)

    print(f"  [{label}] Tam duplicate: {onceki} → {sonra1} ({onceki - sonra1} silindi)")
    print(f"  [{label}] Soru dup.    : {sonra1} → {sonra2} ({sonra1 - sonra2} silindi)")
    return df.reset_index(drop=True)

train_df = remove_duplicates(train_df, "train")
test_df  = remove_duplicates(test_df,  "test")

  [train] Tam duplicate: 13115 → 13115 (0 silindi)
  [train] Soru dup.    : 13115 → 13113 (2 silindi)
  [test] Tam duplicate: 1469 → 1469 (0 silindi)
  [test] Soru dup.    : 1469 → 1469 (0 silindi)


In [9]:
# 7. MEKANİK/KALIP SORU TESPİTİ
# "Anayasa madde X'e göre ... nedir?" kalıpları
# Bunları silmiyoruz ama etiketliyoruz
import re

def detect_mechanical(soru):
    """Mekanik üretilmiş soru kalıplarını tespit et"""
    patterns = [
        r"^Anayasa madde \d+",        # "Anayasa madde 149'a göre..."
        r"madde \d+[''][ae] göre",    # "madde X'e göre"
        r"^Madde \d+",                # "Madde X ..."
    ]
    for p in patterns:
        if re.search(p, soru):
            return True
    return False

train_df["mekanik"] = train_df["soru"].apply(detect_mechanical)
test_df["mekanik"]  = test_df["soru"].apply(detect_mechanical)

mek_train = train_df["mekanik"].sum()
mek_test  = test_df["mekanik"].sum()
print(f"\nMekanik soru tespiti:")
print(f"  Train: {mek_train}/{len(train_df)} ({mek_train/len(train_df)*100:.1f}%)")
print(f"  Test : {mek_test}/{len(test_df)} ({mek_test/len(test_df)*100:.1f}%)")
print(f"\n  Bunlar SİLİNMİYOR — fine-tuning'de korunur.")
print(f"  📌 Gold test seti oluştururken mekanik=False olanlar öncelikli seçilmeli.")


Mekanik soru tespiti:
  Train: 1835/13113 (14.0%)
  Test : 207/1469 (14.1%)

  Bunlar SİLİNMİYOR — fine-tuning'de korunur.
  📌 Gold test seti oluştururken mekanik=False olanlar öncelikli seçilmeli.


In [10]:
#temizleme raporu
print("\n" + "="*60)
print("TEMİZ VERİ RAPORU")
print("="*60)
for label, df in [("TRAIN", train_df), ("TEST", test_df)]:
    print(f"\n[{label}]")
    print(f"  Toplam örnek  : {len(df)}")
    print(f"  Soru uzunluğu : ort={df['soru'].str.len().mean():.0f}, "
          f"min={df['soru'].str.len().min()}, max={df['soru'].str.len().max()}")
    print(f"  Cevap uzunluğu: ort={df['cevap'].str.len().mean():.0f}, "
          f"min={df['cevap'].str.len().min()}, max={df['cevap'].str.len().max()}")


TEMİZ VERİ RAPORU

[TRAIN]
  Toplam örnek  : 13113
  Soru uzunluğu : ort=85, min=15, max=534
  Cevap uzunluğu: ort=200, min=20, max=2542

[TEST]
  Toplam örnek  : 1469
  Soru uzunluğu : ort=84, min=15, max=400
  Cevap uzunluğu: ort=199, min=23, max=2737


In [11]:
#kaydetme
os.makedirs("cleaned_data", exist_ok=True)

train_df.to_json("cleaned_data/hf_train_clean.jsonl",
                 orient="records", lines=True, force_ascii=False)
test_df.to_json("cleaned_data/hf_test_clean.jsonl",
                orient="records", lines=True, force_ascii=False)

print("\n✅ Colab'a kaydedildi:")
print(f"   cleaned_data/hf_train_clean.jsonl  ({len(train_df)} örnek)")
print(f"   cleaned_data/hf_test_clean.jsonl   ({len(test_df)} örnek)")

# Google Drive'a kopyala
print("\nGoogle Drive bağlanıyor...")
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/CENG493/cleaned_data"
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copy("cleaned_data/hf_train_clean.jsonl", f"{DRIVE_DIR}/hf_train_clean.jsonl")
shutil.copy("cleaned_data/hf_test_clean.jsonl",  f"{DRIVE_DIR}/hf_test_clean.jsonl")

print(f"\n Google Drive'a kaydedildi: {DRIVE_DIR}/")
print(f"   hf_train_clean.jsonl  ({len(train_df)} örnek)")
print(f"   hf_test_clean.jsonl   ({len(test_df)} örnek)")

print(f"\n{'='*60}")
print("ÖZET")
print(f"{'='*60}")
print(f"Ham train   : ~13.400 → Temiz train: {len(train_df)}")
print(f"Ham test    : ~1.500  → Temiz test : {len(test_df)}")
print(f"\nSONRAKİ ADIM: Gold test seti ayrıca oluşturulacak.")



✅ Colab'a kaydedildi:
   cleaned_data/hf_train_clean.jsonl  (13113 örnek)
   cleaned_data/hf_test_clean.jsonl   (1469 örnek)

Google Drive bağlanıyor...


NameError: name 'drive' is not defined

In [13]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/CENG493/cleaned_data"
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copy("cleaned_data/hf_train_clean.jsonl", f"{DRIVE_DIR}/hf_train_clean.jsonl")
shutil.copy("cleaned_data/hf_test_clean.jsonl",  f"{DRIVE_DIR}/hf_test_clean.jsonl")

print(f"✅ Google Drive'a kaydedildi: {DRIVE_DIR}/")
print(f"   hf_train_clean.jsonl  (13113 örnek)")
print(f"   hf_test_clean.jsonl   (1469 örnek)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive'a kaydedildi: /content/drive/MyDrive/CENG493/cleaned_data/
   hf_train_clean.jsonl  (13113 örnek)
   hf_test_clean.jsonl   (1469 örnek)
